In [ ]:
import pandas as pd
import time
from tqdm import tqdm
import numpy as np
import os
import logging
import time
import matplotlib.pyplot as plt
import re
from rdkit import Chem
from rdkit.Chem import Descriptors
import pubchempy as pcp
from thermo.chemical import Chemical
import math

In [ ]:
# current_date = time.strftime("%Y%m%d")
# csv_file = f'SelfDiff_md_molecules_{current_date}.csv'
csv_file = 'SelfDiff_MDmolecules'
df = pd.read_csv(f'../data/processed/{csv_file}.csv')
df

In [ ]:
# # Get all molecules after MD_ID 1210
# df = df[df['MD_ID'] >= 1210].reset_index(drop=True)
# df

In [ ]:
def count_atoms(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES string.")
    return mol.GetNumAtoms()

# Example: check atom number for the first molecule in df
smiles_example = df.loc[0, 'can_SMILES']
atom_count = count_atoms(smiles_example)
print(f"SMILES: {smiles_example}\nAtom count: {atom_count}")

# Scan through all molecules and count atoms for each
df['Atom_Count'] = df['can_SMILES'].apply(count_atoms)
df.Atom_Count.describe()

In [ ]:
# Function to generate PDB file from SMILES
def generate_pdb(smiles, molecule_name):
    pdb_file = f"{molecule_name}.pdb"
    if not os.system(f'obabel -:"{smiles}" -O {pdb_file} --gen3d'):
        logging.info(f"Generated {pdb_file} for {molecule_name}")
        return pdb_file
    else:
        logging.error(f"Failed to generate {pdb_file} for {molecule_name}")
        return None

# Function to clean PDB file using pdb4amber
def clean_pdb_with_pdb4amber(pdb_file, molecule_name):
    cleaned_pdb_file = f"{molecule_name}_cleaned.pdb"
    if not os.system(f'pdb4amber -i {pdb_file} -o {cleaned_pdb_file}'):
        logging.info(f"Cleaned {pdb_file} using pdb4amber and saved as {cleaned_pdb_file}")
        return cleaned_pdb_file
    else:
        logging.error(f"Failed to clean {pdb_file} using pdb4amber")
        return None

# Function to run antechamber and parmchk2
def run_antechamber(molecule_name, charge, pdb_file):
    mol2_file = f"{molecule_name}.mol2"
    frcmod_file = f"{molecule_name}.frcmod"
    
    if not os.system(f'antechamber -i {pdb_file} -fi pdb -o {mol2_file} -fo mol2 -c bcc -s 2 -nc {charge} -at gaff2 -rn UNL'):
        logging.info(f"Generated {mol2_file} for {molecule_name}")
    else:
        logging.error(f"Failed to generate {mol2_file} for {molecule_name}")
        return False
    
    if not os.system(f'parmchk2 -i {mol2_file} -f mol2 -o {frcmod_file}'):
        logging.info(f"Generated {frcmod_file} for {molecule_name}")
    else:
        logging.error(f"Failed to generate {frcmod_file} for {molecule_name}")
        return False
    
    return True

def mol2_to_gaff_pdb(molecule_name):
    """
    Convert the GAFF2‐named *.mol2 that antechamber produced into a
    matching *.pdb for Packmol.  Uses tleap via os.system – no subprocess.
    """
    leap_in = f"""\n\
source leaprc.gaff2
UNL = loadmol2 {molecule_name}.mol2
savepdb UNL {molecule_name}_gaff.pdb
quit
"""
    # write a tiny tleap script
    fname = f"{molecule_name}_mol2topdb.in"
    with open(fname, "w") as fh:
        fh.write(leap_in)

    # run tleap; exit code 0 = success
    status = os.system(f"tleap -f {fname}")
    if status != 0 or not os.path.isfile(f"{molecule_name}_gaff.pdb"):
        logging.error(f"Failed to make GAFF PDB for {molecule_name}")
        return None
    return f"{molecule_name}_gaff.pdb"

# Define constant box sizes for benchmark
BOX_SIZES = [35.0]

# Estimate number of molecules in a box
def estimate_molecule_count(density, molar_mass, box_size):
    NA = 6.022e23  # Avogadro's number
    volume_cm3 = (box_size * 1e-8) ** 3  # Convert Å³ to cm³
    N = (density * volume_cm3 * NA) / molar_mass # density in g/cm³, molar mass in g/mol
    return int(np.round(N))

# Function to run Packmol with constant box sizes
def run_packmol(molecule_name, density, molar_mass):
    pdb_file = f"{molecule_name}_gaff.pdb"
    for box_size in BOX_SIZES:
        num_molecules = math.ceil(1.10 * estimate_molecule_count(density, molar_mass, box_size))
        
        packmol_input = f"""
tolerance 2.0
filetype pdb
output {molecule_name}_{int(box_size)}_packed.pdb

structure {pdb_file}
  number {num_molecules}
  add_amber_ter
  connect yes
  inside cube 0. 0. 0. {box_size}
end structure
"""
        with open(f"{molecule_name}_{int(box_size)}_packmol.in", "w") as f:
            f.write(packmol_input)
        
        if not os.system(f"packmol < {molecule_name}_{int(box_size)}_packmol.in"):
            logging.info(
                f"Generated {molecule_name}_{int(box_size)}_packed.pdb "
                f"with {num_molecules} molecules in {box_size} Å box"
            )
        else:
            logging.error(f"Failed to generate {molecule_name}_{int(box_size)}_packed.pdb")
            return False
    return True

# Function to run tleap with constant box sizes
def run_tleap(molecule_name):
    mol2_file = f"{molecule_name}.mol2"
    frcmod_file = f"{molecule_name}.frcmod"
    
    for box_size in BOX_SIZES:
        packed_pdb_file = f"{molecule_name}_{int(box_size)}_packed.pdb"
        prmtop_file = f"{molecule_name}_{int(box_size)}.prmtop"
        inpcrd_file = f"{molecule_name}_{int(box_size)}.inpcrd"
        
        tleap_input = f"""
source leaprc.gaff2
UNL = loadmol2 {mol2_file}
loadamberparams {frcmod_file}
box = loadpdb {packed_pdb_file}
setbox box centers
saveamberparm box {prmtop_file} {inpcrd_file}
quit
"""
        with open(f"{molecule_name}_{int(box_size)}_tleap.in", "w") as f:
            f.write(tleap_input)
        
        if not os.system(f"tleap -f {molecule_name}_{int(box_size)}_tleap.in"):
            logging.info(
                f"Generated {prmtop_file} and {inpcrd_file} for {molecule_name} "
                f"in {box_size} Å box"
            )
        else:
            logging.error(
                f"Failed to generate {prmtop_file} and {inpcrd_file} for {molecule_name} "
                f"in {box_size} Å box"
            )
            return False
    
    return True

In [ ]:
# Main loop
notebook_dir = os.getcwd()  # Get current directory with notebook
output_dir = os.path.join(notebook_dir, '../data/MD/selfdiff/gaff2/systemprep/'+time.strftime("%Y%m%d-%H%M%S"))

# Set up logging to write logs in the notebook's directory
log_filename = os.path.join(os.getcwd(), 'SelfDiff_MDsystemprep_' + time.strftime("%Y%m%d-%H%M%S") + '.log')
logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format='%(asctime)s %(message)s'
)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Change to output directory
os.chdir(output_dir)

for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing molecules"):
    smiles = row["can_SMILES"]
    molecule_name = row["MD_ID"]
    charge = row["Charge"]
    density = row["Density_g_cm3"]
    molar_mass = row["Molar_Mass_g_mol"]
    
    logging.info(f"Processing {molecule_name} ({smiles})...")
    
    # Step 1: Generate PDB file from SMILES
    pdb_file = generate_pdb(smiles, molecule_name)
    if not pdb_file:
        continue
    
    # Step 2: Clean PDB file using pdb4amber
    cleaned_pdb_file = clean_pdb_with_pdb4amber(pdb_file, molecule_name)
    if not cleaned_pdb_file:
        continue
    
    # Step 3: Run antechamber and parmchk2
    if not run_antechamber(molecule_name, charge, cleaned_pdb_file):
        continue
    
    # Step 3.5: Convert the GAFF2‐named *.mol2 that antechamber produced into a matching *.pdb for Packmol
    gaff_pdb = mol2_to_gaff_pdb(molecule_name)
    if gaff_pdb is None:
        continue

    # Step 4: Run Packmol with estimated molecule numbers
    run_packmol(molecule_name, density, molar_mass)

    # Step 5: Run tleap to generate Amber files
    if not run_tleap(molecule_name):
        continue
    
    logging.info(''f"Completed system preparation for {molecule_name}.\n")

# Return to notebook directory
os.chdir(notebook_dir)

In [ ]:
# Restart processing from a certain molecule number (e.g., restart_index)
restart_MD_ID = 445  # Change this to your desired starting index
logging.info(f"Restarting processing from MD_ID {restart_MD_ID}...")

# Change to output directory if needed
os.chdir(output_dir)

for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing molecules"):
    if index+1 < restart_MD_ID:
        continue  # Skip until restart_index

    smiles = row["can_SMILES"]
    molecule_name = row["MD_ID"]
    charge = row["Charge"]
    density = row["Density_g_cm3"]
    molar_mass = row["Molar_Mass_g_mol"]

    logging.info(f"Processing {molecule_name} ({smiles})...")

    pdb_file = generate_pdb(smiles, molecule_name)
    if not pdb_file:
        continue

    cleaned_pdb_file = clean_pdb_with_pdb4amber(pdb_file, molecule_name)
    if not cleaned_pdb_file:
        continue

    if not run_antechamber(molecule_name, charge, cleaned_pdb_file):
        continue

    if not run_packmol(molecule_name, density, molar_mass):
        continue

    if not run_tleap(molecule_name):
        continue

    logging.info(f"Completed system preparation for {molecule_name}.\n")

os.chdir(notebook_dir)